Our preferred attribution is:

Courtesy of the J. Paul Getty Museum, Los Angeles

In addition, most of our written texts are licensed under CC BY, which means that you'll need to correctly attribute the text if you use it. We've included specific attribution content for each text within the data:

In [1]:
import requests
import os
import json

In [7]:
import requests
import re
import json
import os

def extract_endpoints_from_page(url):
    """Extracts REST endpoints from a given webpage using regex."""
    endpoints = []
    response = requests.get(url)
    response.raise_for_status()
    html_content = response.text

    # Define a regex pattern to match REST endpoint URLs
    endpoint_pattern = re.compile(r"https://data\.getty\.edu/museum/collection/(object|person)/[a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12}")

    # Find all matches in the HTML content
    matches = endpoint_pattern.findall(html_content)
    endpoints.extend(matches) 

    return endpoints

def download_entity_data(endpoint_url):
    """Downloads metadata and JPEG for a given entity endpoint."""
    # Download metadata
    metadata_url = f"{endpoint_url}.json"
    response = requests.get(metadata_url)
    response.raise_for_status()
    metadata = response.json()
    entity_id = metadata.get("id", "unknown_id")  # Get entity ID from metadata, or use a default

    # Save metadata as JSON
    with open(f"{entity_id}.json", "w") as f:
        json.dump(metadata, f, indent=4)

    # Download JPEG (assuming it's available at a standard location)
    jpeg_url = f"{endpoint_url}.jpg"  # Adjust this URL based on the API structure
    response = requests.get(jpeg_url, stream=True)
    response.raise_for_status()

    with open(f"{entity_id}.jpg", "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)


# Usage:
starting_url = "https://data.getty.edu/museum/collection/object/e099dc86-c28e-4ad0-8b02-d7d258caec56"
endpoints = extract_endpoints_from_page(starting_url)

for endpoint in endpoints:
    print(endpoint)
    download_entity_data(endpoint)
    print(f"Downloaded data for endpoint: {endpoint}")

object


MissingSchema: Invalid URL 'object.json': No scheme supplied. Perhaps you meant https://object.json?

In [5]:
import requests
import json, re

def extract_entity_ids(entity_type):
    """Extracts entity IDs for a given entity type using regex."""
    ids = []
    offset = 0
    limit = 100  # Number of items to retrieve per request
    id_pattern = re.compile(r"^[a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12}$")

    while True:
        url = f"https://data.getty.edu/museum/collection/{entity_type}?limit={limit}&offset={offset}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if not data.get('items'):
            break

        for item in data.get('items', []):
            potential_id = item.get('id')

            if potential_id and id_pattern.match(potential_id):
                ids.append(potential_id)

        offset += limit

    return ids

def download_entity_data(entity_type, entity_id):
    """Downloads metadata and JPEG for a given entity."""
    base_url = f"https://data.getty.edu/museum/collection/{entity_type}/{entity_id}"

    # Download metadata
    metadata_url = f"{base_url}.json"
    response = requests.get(metadata_url)
    response.raise_for_status()
    metadata = response.json()

    # Save metadata as JSON
    with open(f"{entity_id}.json", "w") as f:
        json.dump(metadata, f, indent=4)

    # Download JPEG (assuming it's available at a standard location)
    jpeg_url = f"{base_url}.jpg"  # You might need to adjust this URL based on the API structure
    response = requests.get(jpeg_url, stream=True)
    response.raise_for_status()

    with open(f"{entity_id}.jpg", "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

# Usage:
for entity_type in ["person", "object"]:
    entity_ids = extract_entity_ids(entity_type)
    for entity_id in entity_ids:
        download_entity_data(entity_type, entity_id)
        print(f"Downloaded data for {entity_type}: {entity_id}")

HTTPError: 404 Client Error: NOT FOUND for url: https://data.getty.edu/museum/collection/person?limit=100&offset=0

In [ ]:
# IIIF API base URL
getty_iiif_base_url = "https://media.getty.edu/iiif/image/"  # Update with the actual base URL if different

# Create a directory to store the images and metadata
if not os.path.exists("/mnt/hdd/maittewa/getty_images"):
    os.makedirs("/mnt/hdd/maittewa/getty_images")

# Function to download and save an image
def download_image(image_url, filename):
    response = requests.get(image_url, stream=True)
    response.raise_for_status()

    filepath = os.path.join("/mnt/hdd/maittewa/getty_images", filename)
    with open(filepath, "wb") as image_file:
        for chunk in response.iter_content(chunk_size=8192):
            image_file.write(chunk)

# Function to fetch image information
def get_image_info(identifier):
    info_url = f"{getty_iiif_base_url}/{identifier}/info.json"  # Construct info.json URL
    response = requests.get(info_url)
    response.raise_for_status()
    return response.json()

# Main loop to process images
for identifier in range(1, 88001):  # Assuming identifiers are sequential
    try:
        # Fetch image information
        image_info = get_image_info(identifier)

        # Extract image URL (customize based on IIIF response)
        image_url = image_info["@id"]  # Assuming "@id" contains the base image URL
        image_url += "/full/full/0/default.jpg"  # Construct full-size image URL

        # Download and save the image
        filename = f"{identifier}.jpg"  # Construct filename
        download_image(image_url, filename)

        # Save image metadata as JSON
        metadata_filepath = os.path.join("/mnt/hdd/maittewa/getty_images_metadata", f"{identifier}.json")
        with open(metadata_filepath, "w") as metadata_file:
            json.dump(image_info, metadata_file, indent=4)

        print(f"Downloaded image and metadata for identifier: {identifier}")

    except requests.exceptions.RequestException as e:
        print(f"Error processing identifier {identifier}: {e}")

In [8]:
import requests
import re
import json
import os

def get_entity_ids(entity_type, query=""):
    """Retrieves entity IDs for a given entity type using the search API."""
    ids = []
    offset = 0
    limit = 100  # Number of items to retrieve per request

    while True:
        url = f"https://data.getty.edu/museum/api/search?query={query}&entity={entity_type}&limit={limit}&offset={offset}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if not data.get('results'):
            break

        for result in data['results']:
            if result.get('type') == entity_type and result.get('id'):
                ids.append(result['id'])

        offset += limit

    return ids

def download_entity_data(entity_type, entity_id):
    """Downloads metadata and JPEG for a given entity."""
    base_url = f"https://data.getty.edu/museum/collection/{entity_type}/{entity_id}"

    # Download metadata
    metadata_url = f"{base_url}.json"
    response = requests.get(metadata_url)
    response.raise_for_status()
    metadata = response.json()

    # Save metadata as JSON
    with open(f"{entity_id}.json", "w") as f:
        json.dump(metadata, f, indent=4)

    # Download JPEG (assuming it's available at a standard location)
    jpeg_url = f"{base_url}.jpg"  # Adjust this URL based on the API structure
    response = requests.get(jpeg_url, stream=True)
    response.raise_for_status()

    with open(f"{entity_id}.jpg", "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

# Usage:
for entity_type in ["person", "object"]:
    entity_ids = get_entity_ids(entity_type)  # Get IDs for the entity type
    for entity_id in entity_ids:
        download_entity_data(entity_type, entity_id)
        print(f"Downloaded data for {entity_type}: {entity_id}")

HTTPError: 503 Server Error: Service Temporarily Unavailable for url: https://data.getty.edu/museum/api/search?query=&entity=person&limit=100&offset=0

In [11]:
import requests
import re
import json
import os

def get_entity_url(entity_type, entity_id):
    """Constructs the URL for a given entity."""
    return f"https://data.getty.edu/museum/collection/{entity_type}/{entity_id}.json"

def crawl_getty_entities(starting_url, entity_types=["object", "person"], visited_urls=None):
    """Recursively crawls Getty Museum entities and extracts their URLs."""
    if visited_urls is None:
        visited_urls = set()

    visited_urls.add(starting_url)
    entity_urls = [starting_url]  # Start with the initial URL

    try:
        response = requests.get(starting_url)
        response.raise_for_status()
        data = response.json()

        # Extract URLs for related entities
        for entity_type in entity_types:
            for related_entity in data.get(entity_type, []): 
                related_entity_id = related_entity.get("id")
                if related_entity_id:
                    related_entity_url = get_entity_url(entity_type, related_entity_id)
                    if related_entity_url not in visited_urls:
                        entity_urls.extend(crawl_getty_entities(related_entity_url, entity_types, visited_urls))

    except requests.exceptions.RequestException as e:
        print(f"Error fetching or processing URL: {starting_url} - {e}")

    return entity_urls

# Usage:
starting_url = "https://media.getty.edu/iiif/manifest/5691c0b8-c007-49bf-be14-8ca8aacd2e9d"  # Example starting URL
all_entity_urls = crawl_getty_entities(starting_url)

# Print or process the collected URLs
for url in all_entity_urls:
    print(url)
    # ... (Download data or perform other actions with the URL) ...

https://media.getty.edu/iiif/manifest/5691c0b8-c007-49bf-be14-8ca8aacd2e9d


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json
import os

def get_entity_url(entity_type, entity_id):
    """Constructs the URL for a given entity."""
    return f"https://data.getty.edu/museum/collection/{entity_type}/{entity_id}.json"

def extract_entity_urls_from_search_page(search_url):
    """Extracts entity URLs from a Getty search results page."""
    entity_urls = []
    response = requests.get(search_url)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")    
    # Find links to object and person pages
    for link in soup.find_all("a", href=True):
        href = link["href"]
        if "/art/collection/objects/" in href or "/art/collection/artists/" in href:
            # Extract entity type and ID from the URL
            match = re.search(r"/(objects|artists)/([a-zA-Z0-9-]+)", href)
            if match:
                entity_type = match.group(1)  # "objects" or "artists"
                entity_id = match.group(2)
                entity_url = get_entity_url(entity_type, entity_id)
                entity_urls.append(entity_url)

    return entity_urls

def crawl_getty_entities(starting_url, entity_types=["object", "person"], visited_urls=None):
    """Recursively crawls Getty Museum entities and extracts their URLs."""
    # ... (This function remains the same as in the previous response) ...

# Usage:
search_url = "https://www.getty.edu/art/collection/search?open_content=true&date_range=1840:2010"
initial_entity_urls = extract_entity_urls_from_search_page(search_url)

all_entity_urls = []
for url in initial_entity_urls:
    all_entity_urls.extend(crawl_getty_entities(url))

# Print or process the collected URLs
for url in all_entity_urls:
    print(url)
    # ... (Download data or perform other actions with the URL) ...

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json
import os

# ... (Previous functions: get_entity_url, crawl_getty_entities) ...

def extract_entity_urls_from_search(search_url):
    """Extracts entity URLs from all pages of Getty search results."""
    all_entity_urls = []
    current_page = 1

    while True:
        # Construct the URL for the current page
        page_url = f"{search_url}&page={current_page}" 

        response = requests.get(page_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        # Extract entity URLs from the current page
        page_entity_urls = []
        for div in soup.find_all("div", class_="a-image"):
            link = div.find("#app > div > div > main > section:nth-child(2) > div > div > section > div.p-search__search-results > div.m-grid-of-items.g-align-image-base--grid > ul > li:nth-child(1) > div > div > a > div.a-image > div > img")
            if link:
                href = link["href"]
                match = re.search(r"/(objects|artists)/([a-zA-Z0-9-]+)", href)
                if match:
                    entity_type = match.group(1)
                    entity_id = match.group(2)
                    entity_url = get_entity_url(entity_type, entity_id)
                    page_entity_urls.append(entity_url)

        # Add the extracted URLs to the overall list
        all_entity_urls.extend(page_entity_urls)

        # Check if there's a next page
        next_page_link = soup.find("a", class_="next-page")  # Adjust class if needed
        if not next_page_link:
            break  # No next page, stop the loop

        current_page += 1

    return all_entity_urls

# Usage:
search_url = "https://www.getty.edu/art/collection/search?open_content=true&date_range=1840:2010"
all_entity_urls = extract_entity_urls_from_search(search_url)

# Print or process the collected URLs
for url in all_entity_urls:
    print(url)
    # ... (Download data or perform other actions with the URL) ...

In [23]:
import requests
from bs4 import BeautifulSoup
import os

def download_image(image_url, output_dir):
    """Downloads an image and saves it to the specified path."""
    response = requests.get(image_url, stream=True)
    response.raise_for_status()
    if response is not None:
        filename = f"{deviation.title}-{deviation.deviationid}.jpg"
        filepath = os.path.join(output_dir, deviant, filename)
        os.makedirs(os.path.dirname(filepath), exist_ok=True) 
    with open(filepath, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

def extract_and_download_images(search_url):
    """Extracts image URLs and downloads them."""
    response = requests.get(search_url)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    print(soup.prettify())
    # Find all divs with the specified class
    for div in soup.find_all("div.a-image"):  # Adjust class if needed
        # Find the img tag within the div
        if div:
            img_tag = div.find("img", class_ = "a-image__img", src=True)
            if img_tag:
                image_url = img_tag["src"]
            
                # Construct the filename (e.g., using the last part of the URL)
                filename = os.path.basename(image_url)
                save_path = os.path.join("/home/maittewa/codes/GettyMuseum/images", filename)  # Create an "images" folder

                # Download and save the image
                download_image(image_url, save_path)
                print(f"Downloaded image: {filename}")
            else:
                print("No image tag")
        else:
            print("No divs")
        

# Usage:
search_url = "https://www.getty.edu/art/collection/search?open_content=true&date_range=1840:2010"  # Your search URL
extract_and_download_images(search_url)

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <link href="/art/collection/favicon.ico" rel="icon"/>
  <meta content="Explore the collection of the J. Paul Getty Museum at the Getty Center and the Getty Villa." name="description"/>
  <meta content="Getty Museum collections, artworks, art works, art history, video, interpretation of art, acquisitions, artists, Getty art, collections Getty, collezioni Getty, colecciones Getty, Sammlung Getty, Getty exhibits, Getty current exhibitions, Getty Museum exhibits, los angeles collections, antiquities, decorative arts, sculpture, manuscripts, photography, paintings, drawings" name="keywords"/>
  <!-- share -->
  <script id="local_id_manager" type="application/json">
   {"slug": null, "indexedId": null}
  </script>
  <meta content="The J. Paul Getty Museum Collection" name="og:site_name"/>
  <meta c

In [30]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import requests
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

output_dir = f"/mnt/hdd/maittewa/gettyMuseumGallery/downloaded_gallery_{deviant}"
os.makedirs(output_dir, exist_ok=True)

# Update ChromeOptions to specify headless mode
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# Initialize Chrome WebDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Load the webpage
search_url = "https://www.getty.edu/art/collection/search?open_content=true&date_range=1840:2010"
driver.get(search_url)

# Wait for the page to fully load (adjust the time if needed)
time.sleep(10)  # Wait for 10 seconds (or use more robust methods)

# Get the page source after JavaScript has executed
html = driver.page_source

# Parse with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
print(soup.prettify())
for div in soup.find_all("div", class_="a-image"):  # Adjust class if needed
        # Find the img tag within the div
        if div:
            img_tag = div.find("img", class_="a-image__img")
            if img_tag:
                image_url = img_tag["src"]
            
                # Construct the filename (e.g., using the last part of the URL)
                filename = os.path.basename(image_url)
                save_path = os.path.join("/home/maittewa/codes/GettyMuseum/images", filename)  # Create an "images" folder

                # Download and save the image
                download_image(image_url, save_path)
                print(f"Downloaded image: {filename}")
            else:
                print("No image tag")
        else:
            print("No divs")
# Close the browser
driver.close()

<html lang="en" style="--vh: 6px;">
 <head>
  <style class="vjs-styles-defaults">
   .video-js {
        width: 300px;
        height: 150px;
      }

      .vjs-fluid:not(.vjs-audio-only-mode) {
        padding-top: 56.25%
      }
  </style>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <link href="/art/collection/favicon.ico" rel="icon"/>
  <meta content="Explore the collection of the J. Paul Getty Museum at the Getty Center and the Getty Villa." name="description"/>
  <meta content="Getty Museum collections, artworks, art works, art history, video, interpretation of art, acquisitions, artists, Getty art, collections Getty, collezioni Getty, colecciones Getty, Sammlung Getty, Getty exhibits, Getty current exhibitions, Getty Museum exhibits, los angeles collections, antiquities, decorative arts, sculpture, manuscripts, photography, paintings, drawings" name="keywords"/>
  <!--

FileNotFoundError: [Errno 2] No such file or directory: '/home/maittewa/codes/GettyMuseum/images/default.jpg'